# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
# ============================================================
# ML-05 — DATA AVAILABILITY DIAGNOSTICS (consolidated)
# ============================================================

from google.colab import userdata
import os
import duckdb
import pandas as pd
import numpy as np

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

daily_path = f"{rel}/fact_content_daily_performance/month=*/data_0.parquet"


# ------------------------------------------------------------
# CHECK 0: Full schema of the daily fact table
# (Confirms every column that exists — run this first, before
# trusting any completeness check below to be "complete")
# ------------------------------------------------------------
schema = con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{daily_path}')
""").df()
print("=== FULL SCHEMA: daily performance table ===")
print(schema)


# ------------------------------------------------------------
# CHECK 0B: Required columns for model dataframe construction
# ------------------------------------------------------------
required_model_columns = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "scroll_events"
]

available_columns = set(schema["column_name"])

missing_model_columns = [
    col for col in required_model_columns
    if col not in available_columns
]

print("\n=== CHECK 0B: Required model columns ===")
print("Required model columns:", len(required_model_columns))
print("Columns present:", len(required_model_columns) - len(missing_model_columns))
print("Columns missing:", len(missing_model_columns))

if missing_model_columns:
    print("Missing columns:")
    print(missing_model_columns)
else:
    print("All required model columns are present.")


# ------------------------------------------------------------
# CHECK 1: Unique clients overall
# ------------------------------------------------------------
unique_clients = con.sql(f"""
    SELECT COUNT(DISTINCT client_hash_id) AS unique_clients
    FROM read_parquet('{daily_path}')
""").df()
print("\n=== CHECK 1: Unique clients ===")
print(unique_clients)


# ------------------------------------------------------------
# CHECK 2: Unique (client, page) combinations overall
# ------------------------------------------------------------
unique_pages = con.sql(f"""
    SELECT COUNT(DISTINCT (client_hash_id, content_hash_id)) AS unique_client_page_pairs
    FROM read_parquet('{daily_path}')
""").df()
print("\n=== CHECK 2: Unique (client, page) pairs ===")
print(unique_pages)


# ------------------------------------------------------------
# CHECK 3: Feature completeness on the known GSC columns
# (only checks the 3 columns already selected — NOT every
# column in the table; see Check 0 for the full list)
# ------------------------------------------------------------
completeness_check = con.sql(f"""
    SELECT
        COUNT(DISTINCT (client_hash_id, content_hash_id)) AS total_pairs,
        COUNT(DISTINCT CASE
            WHEN gsc_impressions IS NOT NULL
             AND gsc_clicks IS NOT NULL
             AND gsc_avg_position > 0
            THEN (client_hash_id, content_hash_id)
        END) AS pairs_with_at_least_one_complete_row
    FROM read_parquet('{daily_path}')
""").df()
print("\n=== CHECK 3: Completeness on gsc_impressions / gsc_clicks / gsc_avg_position ===")
print(completeness_check)


# ------------------------------------------------------------
# CHECK 4: Total unique pages + reporting-day thresholds
# ------------------------------------------------------------
page_summary = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(DISTINCT report_date) AS total_reported_days
    FROM read_parquet('{daily_path}')
    GROUP BY client_hash_id, content_hash_id
""").df()

total_pages = len(page_summary)
pages_120_or_more = (page_summary["total_reported_days"] >= 120).sum()
pages_less_than_120 = (page_summary["total_reported_days"] < 120).sum()

print("\n=== CHECK 4: Total pages + reporting-day thresholds ===")
print("Total unique pages:", total_pages)
print("Pages with >= 120 reporting days:", pages_120_or_more)
print("Pages with < 120 reporting days:", pages_less_than_120)


# ------------------------------------------------------------
# CHECK 5: Monthly reporting-day coverage distribution
# ------------------------------------------------------------
monthly_page_days = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        DATE_TRUNC('month', report_date) AS month,
        COUNT(DISTINCT report_date) AS reported_days
    FROM read_parquet('{daily_path}')
    GROUP BY client_hash_id, content_hash_id, DATE_TRUNC('month', report_date)
""").df()

monthly_page_days["coverage_group"] = pd.cut(
    monthly_page_days["reported_days"],
    bins=[-1, 0, 10, 20, 27, 29, 30, 31],
    labels=["0 days", "1-10 days", "11-20 days", "21-27 days",
            "28-29 days", "30 days", "31 days"]
)

monthly_coverage_summary = (
    monthly_page_days
    .groupby(["month", "coverage_group"], observed=False)
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

print("\n=== CHECK 5: Monthly reporting-day coverage ===")
print(monthly_coverage_summary)


# ------------------------------------------------------------
# CHECK 6: Pages meeting monthly reporting-day thresholds
# ------------------------------------------------------------
pages_20_plus_per_month = (
    monthly_page_days
    .groupby("month")
    .agg(
        total_pages=("content_hash_id", "nunique"),
        pages_with_20_plus_days=("reported_days", lambda x: (x >= 20).sum()),
        pages_with_28_plus_days=("reported_days", lambda x: (x >= 28).sum()),
        pages_with_30_plus_days=("reported_days", lambda x: (x >= 30).sum())
    )
    .reset_index()
)

print("\n=== CHECK 6: Pages meeting monthly reporting-day thresholds ===")
print(pages_20_plus_per_month)

=== FULL SCHEMA: daily performance table ===
                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users   

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== CHECK 1: Unique clients ===
   unique_clients
0              70


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== CHECK 2: Unique (client, page) pairs ===
   unique_client_page_pairs
0                    427292


...


=== CHECK 3: Completeness on gsc_impressions / gsc_clicks / gsc_avg_position ===
   total_pairs  pairs_with_at_least_one_complete_row
0       427292                                295604


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== CHECK 4: Total pages + reporting-day thresholds ===
Total unique pages: 427292
Pages with >= 120 reporting days: 309925
Pages with < 120 reporting days: 117367


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== CHECK 5: Monthly reporting-day coverage ===
coverage_group      month  0 days  1-10 days  11-20 days  21-27 days  \
0              2025-01-01       0        476           0           0   
1              2025-02-01       0       2117        2656        1041   
2              2025-03-01       0       4917        1252         606   
3              2025-04-01       0       2278        2562        1413   
4              2025-05-01       0       2517        2156        1472   
5              2025-06-01       0       4435        2716        1294   
6              2025-07-01       0       9606        7866        1923   
7              2025-08-01       0      12187        5006        5109   
8              2025-09-01       0      23754        6864        6449   
9              2025-10-01       0      16371       59259        7740   
10             2025-11-01       0       2684        2246      153449   
11             2025-12-01       0       2901        3459        1684   
12             

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.